In [ ]:
# =============================
# 1. Imports
# =============================
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# =============================
# 2. Load data (MNIST)
# =============================
# (x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()

# To demonstrate a manual train-test split, we'll combine and then split ourselves
(x_train_full, y_train_full), (x_test_full, y_test_full) = mnist.load_data()

X = np.concatenate([x_train_full, x_test_full], axis=0)
y = np.concatenate([y_train_full, y_test_full], axis=0)

print("Original data shape:", X.shape, y.shape)

# =============================
# 3. Train-test split (e.g., 80/20)
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# =============================
# 4. Preprocessing
# =============================

# Reshape to (samples, height, width, channels)
# MNIST is 28x28 grayscale → channels = 1
X_train = X_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# One-hot encode labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# =============================
# 5. Define CNN model
# =============================
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.summary()

# =============================
# 6. Compile model
# =============================
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# =============================
# 7. Train model (with validation split)
# =============================
history = model.fit(
    X_train, y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# =============================
# 8. Evaluate on test data
# =============================
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# =============================
# 9. Predictions & performance metrics
# =============================
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =============================
# 10. Plot training curves
# =============================
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

# =============================
# 11. Visual check: some predictions
# =============================
def show_samples(X, y_true, y_pred, n=10):
    plt.figure(figsize=(15, 3))
    indices = np.random.choice(len(X), n, replace=False)
    for i, idx in enumerate(indices):
        img = X[idx].reshape(28, 28)
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
        plt.title(f"T:{y_true[idx]} P:{y_pred[idx]}")
    plt.show()

show_samples(X_test, y_test, y_pred, n=10)
